In [1]:
import pandas as pd
import numpy as np
import h3

In [2]:
file = '../datasets/database_processed_h3.csv'

In [3]:
df_raw = pd.read_csv(file)

In [4]:
df_raw['DATA'] = pd.to_datetime(df_raw['DATA'], format='%m-%Y')

In [7]:
cols = ['FURTO', 'ROUBO', 'CNT_CRIME']
meses = pd.date_range(df_raw['DATA'].min(), df_raw['DATA'].max(), freq='MS')
grade = pd.MultiIndex.from_product([df_raw['H3'].unique(), meses], names=['H3', 'DATA'])

In [8]:
df_raw = (df_raw.groupby(['H3', 'DATA'])[cols].sum()
                .reindex(grade, fill_value=0)
                .reset_index()
                .sort_values(['H3', 'DATA'])
                .reset_index(drop=True))

In [9]:
df_raw = df_raw.sort_values(['H3', 'DATA']).reset_index(drop=True)

In [10]:
celulas = df_raw['H3'].unique()
cset = set(celulas)
viz = {c: [v for v in h3.grid_disk(c, 1) if v != c and v in cset] for c in celulas}

In [11]:
def create_features(base, col, pref):
    d = base[['H3', 'DATA', col]].copy()
    d['ocorr'] = (d[col] > 0).astype(float)

    g = d.groupby('H3')
    d['p'] = g[col].shift(1)           
    d['po'] = g['ocorr'].shift(1)
    gp = d.groupby('H3')

    media_3m = gp['p'].transform(lambda s: s.rolling(3, min_periods=3).mean())
    media_12m = gp['p'].transform(lambda s: s.rolling(12, min_periods=12).mean())
    lag_12 = g[col].shift(12)

    out = d[['H3', 'DATA']].copy()
    out[f'{pref}_media_12m_log'] = np.log1p(media_12m)
    out[f'{pref}_razao_3_12'] = (media_3m + 0.5) / (media_12m + 0.5)
    out[f'{pref}_sazonal_12'] = (lag_12 + 0.5) / (media_12m + 0.5)
    out[f'{pref}_pct_meses_12m'] = gp['po'].transform(lambda s: s.rolling(12, min_periods=12).mean())

    wide = d.pivot(index='DATA', columns='H3', values=col).sort_index()
    viz_media = pd.DataFrame(
        {c: (wide[v].mean(axis=1) if v else 0.0) for c, v in viz.items()},
        index=wide.index)
    viz_3m = np.log1p(viz_media.shift(1).rolling(3, min_periods=3).mean())
    viz_3m = viz_3m.rename_axis(columns='H3') 
    viz_long = (viz_3m.stack(future_stack=True)
                      .rename(f'{pref}_viz_media_3m_log')
                      .reset_index())
    return out.merge(viz_long, on=['DATA', 'H3'], how='left')

In [12]:
alvos = {'crime': 'CNT_CRIME', 'furto': 'FURTO', 'roubo': 'ROUBO'}

In [13]:
for pref, col in alvos.items():
    df_raw = df_raw.merge(create_features(df_raw, col, pref), on=['H3', 'DATA'], how='left')

In [23]:
sufixos = ['media_12m_log', 'razao_3_12', 'sazonal_12', 'pct_meses_12m', 'viz_media_3m_log', 'viz_rel']
feats = {pref: [f'{pref}_{s}' for s in sufixos] for pref in alvos}

In [24]:
for pref in ['crime', 'furto', 'roubo']:
    df_raw[f'{pref}_viz_rel'] = (df_raw[f'{pref}_viz_media_3m_log']
                                 - df_raw[f'{pref}_media_12m_log'])

In [25]:
train = df_raw[df_raw['DATA'].dt.year > 2022]

In [26]:
for pref, cols in feats.items():
    print(f'\n== {pref} ==')
    print(train[cols].corr(method='spearman').round(2))


== crime ==
                        crime_media_12m_log  crime_razao_3_12  \
crime_media_12m_log                    1.00             -0.01   
crime_razao_3_12                      -0.01              1.00   
crime_sazonal_12                       0.10             -0.14   
crime_pct_meses_12m                    0.96             -0.01   
crime_viz_media_3m_log                 0.87              0.01   
crime_viz_rel                         -0.19              0.02   

                        crime_sazonal_12  crime_pct_meses_12m  \
crime_media_12m_log                 0.10                 0.96   
crime_razao_3_12                   -0.14                -0.01   
crime_sazonal_12                    1.00                 0.10   
crime_pct_meses_12m                 0.10                 1.00   
crime_viz_media_3m_log              0.12                 0.84   
crime_viz_rel                      -0.04                -0.13   

                        crime_viz_media_3m_log  crime_viz_rel  
crime_media

In [27]:
sufixos = ['media_12m_log', 'razao_3_12', 'sazonal_12', 'viz_rel']
feats = {pref: [f'{pref}_{s}' for s in sufixos] for pref in alvos}

In [28]:
df_raw = df_raw[feats['crime'] + feats['furto'] + feats['roubo'] + ['H3', 'DATA', 'CNT_CRIME', 'ROUBO', 'FURTO']]

In [30]:
df_raw.to_csv('../datasets/database_processed_h3_with_features.csv', index=False)